## Classifier code

In [2]:
#Import statements
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, balanced_accuracy_score, auc, accuracy_score, recall_score, f1_score
from sklearn.model_selection import cross_val_predict, LeaveOneOut, train_test_split
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
import matplotlib.lines as mlines
print(torch.__version__)

2.3.0


#### Input non-labeled training, testing datasets, and diagnosis-labeled test data

In [ ]:
train_csv = pd.read_csv('model_data/train_brain_labels_deid.csv', index_col=0)
train_parquet_scaled = pd.read_parquet('model_data/train_brain_data_deid_scaled.parquet')
test_csv = pd.read_csv('model_data/test_brain_labels_deid.csv', index_col=0)
test_parquet_scaled = pd.read_parquet('model_data/test_brain_data_deid_scaled.parquet')
nd_csv = pd.read_csv('model_data/nd_brain_labels_deid.csv', index_col=0)
nd_parquet_scaled = pd.read_parquet('model_data/nd_brain_data_deid_scaled.parquet')

#### Input shortened labeled dataset

Retain only data with diagnosis of ad, bvftd, cu, or dlb, and age and sex information

In [ ]:
nd_csv_short = pd.read_csv('model_data/nd_filtered_data.csv', index_col=0)

#### Remove sex label

In [ ]:
desired_columns = ['age_at_scan', 'ad', 'bvftd', 'cu', 'dlb']
nd_csv_nsex = nd_csv_short[desired_columns].copy()
nd_removed_columns = list(set(nd_csv_nsex.columns) - set(desired_columns))
for col in nd_removed_columns:
    if col in nd_csv_short:
        mask = (nd_csv_nsex[col] != 1)
        mask = mask.loc[nd_csv_nsex.index]
        nd_csv_nsex = nd_csv_nsex[mask]

In [ ]:
nd_csv_nage = nd_csv_nsex.drop(axis=1, columns='age_at_scan')
nd_csv_full = nd_csv.drop(axis=1, columns=['age_at_scan', 'sex_female'])

In [ ]:
x_data = pd.read_csv('model_data/model_16dim_full_embeddings_scaled.csv')
x_data = x_data.drop(axis=1, columns='Unnamed: 0')

In [ ]:
le = LabelEncoder()
y = le.fit_transform(nd_csv_full.idxmax(axis=1))
X = x_data.values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

#### Logistic Regression

In [ ]:
lr_vae = LogisticRegression(C=10, solver='liblinear')
lr_vae.fit(X_train, y_train)

In [ ]:
y_pred_proba = lr_vae.predict_proba(X_test)
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print(f"ROC AUC Score: {roc_auc}")

#### Calculate combined balanced accuracy

In [ ]:
y_pred = lr_vae.predict(X_test)
total_balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
print(f'Total combined balanced accuracy: {total_balanced_accuracy:.2f}')

In [ ]:
classes = np.unique(y_test)
y_bin = label_binarize(y_test, classes=classes)

# Predict probabilities for each class
y_pred_proba = lr_vae.predict_proba(X_test)

# Calculate ROC AUC for each class
roc_auc_scores = {}
for i, class_label in enumerate(classes):
    roc_auc_scores[le.inverse_transform([class_label])[0]] = roc_auc_score(y_bin[:, i], y_pred_proba[:, i])

# Print ROC AUC scores for each class
for label, score in roc_auc_scores.items():
    print(f"ROC AUC Score for {label}: {score}")

In [ ]:
ad_labels = nd_csv['ad'].copy()
bvftd_labels = nd_csv['bvftd'].copy()
cbs_labels = nd_csv['cbs'].copy()
cu_labels = nd_csv['cu'].copy()
dlb_labels = nd_csv['dlb'].copy()
lvppa_labels = nd_csv['lvppa'].copy()
nfppa_labels = nd_csv['nfppa'].copy()
pca_labels = nd_csv['pca'].copy()
ppaos_labels = nd_csv['ppaos'].copy()
psp_labels = nd_csv['psp'].copy()
sd_labels = nd_csv['sd'].copy()

dementia_types = ['ad', 'bvftd', 'cbs', 'cu', 'dlb', 'lvppa', 'nfppa', 'pca', 'ppaos', 'psp', 'sd']
labels = [ad_labels, bvftd_labels, cbs_labels, cu_labels, dlb_labels, lvppa_labels, nfppa_labels, pca_labels, ppaos_labels, psp_labels, sd_labels]

In [ ]:
color_dict = {'ad' : "#006400",'bvftd' : "#FF7F50",'cbs' : 'pink', 'cu' : "#838B8B",'dlb' : "#ffd700",'lvppa' : "#00ff00",'nfppa' : "#8B4513",'pca' : "#00ffff",'ppaos' : "#ff1493",'psp' : "#a020f0",'sd' : "#1e90ff", 'total' : 'black'}

#### Plot ROC curve for each dementia type

In [ ]:
plt.figure(figsize=(4, 6))

for dementia_type, label in zip(dementia_types, labels):
    y_pred_proba = lr_vae.predict_proba(X)[:, le.transform([dementia_type])[0]]
    fpr, tpr, _ = roc_curve(label, y_pred_proba)
    roc_auc = roc_auc_score(label, y_pred_proba)
    plt.plot(fpr, tpr, lw=2, label=f'{dementia_type} ROC curve (area = {roc_auc:.2f})', color=color_dict[dementia_type])

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve -- Logistic Regression Model')
plt.legend(fontsize="10", loc="lower right")
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right']. set_visible(False)
plt.show()

#### K Nearest Neighbors

In [ ]:
knn_vae = KNeighborsClassifier(n_neighbors=8, metric='cosine')
knn_vae.fit(X_train, y_train)

In [ ]:
y_pred_proba = knn_vae.predict_proba(X_test)
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print(f"ROC AUC Score: {roc_auc}")

#### Plot ROC curve for each dementia type

In [ ]:
plt.figure(figsize=(4, 6))

for dementia_type, label in zip(dementia_types, labels):
    y_pred_proba = knn_vae.predict_proba(X)[:, le.transform([dementia_type])[0]]
    fpr, tpr, _ = roc_curve(label, y_pred_proba)
    roc_auc = roc_auc_score(label, y_pred_proba)
    plt.plot(fpr, tpr, lw=2, label=f'{dementia_type} ROC curve (area = {roc_auc:.2f})', color=color_dict[dementia_type])

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve -- KNN Model')
plt.legend(fontsize="10", loc="lower right")
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right']. set_visible(False)
plt.show()

#### KNN and LR Ensemble model

In [ ]:
knn_ens = KNeighborsClassifier(n_neighbors=8, metric='cosine')
knn_ens.fit(X_train, y_train)
lr_vae_ens = LogisticRegression(C=10, solver='liblinear')
lr_vae_ens.fit(X_train, y_train)
y_pred_proba_knn_ens = knn_ens.predict_proba(X_test)
y_pred_proba_lr_ens = lr_vae_ens.predict_proba(X_test)
y_pred_proba_ensemble = (y_pred_proba_knn_ens + y_pred_proba_lr_ens) / 2

In [ ]:
roc_auc_ens = roc_auc_score(y_test, y_pred_proba_ensemble, multi_class='ovr')
print(f"Ensemble ROC AUC Score: {roc_auc_ens}")

In [ ]:
classes = np.unique(y_test)
y_bin = label_binarize(y_test, classes=classes)

y_pred_proba_knn = knn_ens.predict_proba(X_test)
y_pred_proba_lr = lr_vae_ens.predict_proba(X_test)
y_pred_proba_ensemble = (y_pred_proba_knn + y_pred_proba_lr) / 2

roc_auc_scores = {}
for i, class_label in enumerate(classes):
    roc_auc_scores[le.inverse_transform([class_label])[0]] = roc_auc_score(y_bin[:, i], y_pred_proba_ensemble[:, i])
    
for label, score in roc_auc_scores.items():
    print(f"Ensemble ROC AUC Score for {label}: {score}")

#### Plot ROC curve for each dementia type

In [ ]:
plt.figure(figsize=(4, 6))

for dementia_type, label in zip(dementia_types, labels):
    y_pred_proba_knn = knn_ens.predict_proba(X)[:, le.transform([dementia_type])[0]]
    y_pred_proba_lr = lr_vae_ens.predict_proba(X)[:, le.transform([dementia_type])[0]]
    y_pred_proba_ensemble = (y_pred_proba_knn + y_pred_proba_lr) / 2
    
    fpr, tpr, _ = roc_curve(label, y_pred_proba_ensemble)
    roc_auc = roc_auc_score(label, y_pred_proba_ensemble)
    plt.plot(fpr, tpr, lw=2, label=f'{dementia_type} ROC curve (area = {roc_auc:.2f})', color=color_dict[dementia_type])

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve -- Ensemble Model')
plt.legend(fontsize="10", loc="lower right")
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right']. set_visible(False)
plt.show()

#### Voting Classifier

In [ ]:
y_binary = np.where(y == 3, 1, 0)  # 1 for label, 0 for others
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, shuffle=True)
knn = KNeighborsClassifier(n_neighbors=8, metric='cosine')
lr = LogisticRegression(C=10, solver='liblinear')
ensemble_model = VotingClassifier(estimators=[('knn', knn), ('lr', lr)], voting='soft')
ensemble_model.fit(X_train, y_train)

In [ ]:
y_pred = ensemble_model.predict(X_test)
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Weighted F1 Score: {weighted_f1:.2f}")
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

#### AD vs CU classification

In [ ]:
filter_mask = np.isin(y, [0, 3])
X_ac = X[filter_mask]
y_ac = y[filter_mask]
y_ac = np.where(y_ac == 3, 1, 0)
X_train, X_test, y_train, y_test = train_test_split(X_ac, y_ac, test_size=0.2, shuffle=True)

In [ ]:
lr_vae = LogisticRegression(C=10, solver='liblinear')
lr_vae.fit(X_train, y_train)

# Predict probabilities for the test set
y_pred_proba = lr_vae.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.2f}")

In [ ]:
y_pred = lr_vae.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"Macro F1 Score: {macro_f1}")
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
sensitivity = recall_score(y_test, y_pred, average='macro') 
print(f"Sensitivity (Recall): {sensitivity}")

#### PCA version

In [ ]:
mask_indices = nd_csv.index
nd_data_org_short = nd_parquet_scaled
pca_res = PCA(n_components=16, whiten=True).fit_transform(nd_data_org_short)

In [ ]:
le = LabelEncoder()
y = le.fit_transform(nd_csv_full.idxmax(axis=1))
X_pca = pca_res
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, shuffle=True)
lr_vae = LogisticRegression(C=10, solver='liblinear')
lr_vae.fit(X_train, y_train)

In [ ]:
knn_pca = KNeighborsClassifier(n_neighbors=8, metric='cosine')
knn_pca.fit(X_train_pca, y_train_pca)

In [ ]:
y_pred_proba = knn_pca.predict_proba(X_test_pca)
roc_auc = roc_auc_score(y_test_pca, y_pred_proba, multi_class='ovr')
print(f"ROC AUC Score: {roc_auc}")

In [ ]:
classes = np.unique(y_test_pca)
y_bin = label_binarize(y_test_pca, classes=classes)
y_pred_proba = knn_pca.predict_proba(X_test_pca)

roc_auc_scores = {}
for i, class_label in enumerate(classes):
    roc_auc_scores[le.inverse_transform([class_label])[0]] = roc_auc_score(y_bin[:, i], y_pred_proba[:, i])

for label, score in roc_auc_scores.items():
    print(f"ROC AUC Score for {label}: {score}")

Now each column corresponds to a binary label for each class

In [ ]:
dementia_types = ['ad', 'bvftd', 'cu', 'dlb']
y_test_one_hot = pd.get_dummies(y_test_pca, prefix='class')
y_test_one_hot.columns = dementia_types
y_test_one_hot = y_test_one_hot.astype(int)

# Now each column corresponds to a binary label for each class
ad_labels_test = y_test_one_hot['ad']
bvftd_labels_test = y_test_one_hot['bvftd']
cu_labels_test = y_test_one_hot['cu']
dlb_labels_test = y_test_one_hot['dlb']

labels = [ad_labels_test, bvftd_labels_test, cu_labels_test, dlb_labels_test]

In [ ]:
plt.figure(figsize=(6, 6))

for dementia_type, label in zip(dementia_types, labels):
    y_pred_proba = knn_pca.predict_proba(X_pca)[:, le.transform([dementia_type])[0]]
    fpr, tpr, _ = roc_curve(label, y_pred_proba)
    roc_auc = roc_auc_score(label, y_pred_proba)
    plt.plot(fpr, tpr, lw=2, label=f'{dementia_type} ROC curve (area = {roc_auc:.2f})', color=color_dict[dementia_type])

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve -- PCA Model')
plt.legend(fontsize="12", loc="lower right")
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right']. set_visible(False)
plt.show()